# Preprocesamiento Completo - Statlog (Australian Credit Approval)

En este cuadernillo se aplica el flujo completo de preprocesamiento con estilo de los cuadernillos del inge:
1. Carga y descripcion de datos
2. Split 80/20
3. Tratamiento de nulos
4. Codificacion
5. Normalizacion con featureNormalize
6. Balanceo del train
7. Guardado de salidas

In [60]:
# ==========================================
# 1. IMPORTACION DE LIBRERIAS
# ==========================================
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

%matplotlib inline

## Paso 1: Carga del dataset
Se carga `australian.dat` (15 columnas: 14 atributos y 1 clase).

In [61]:
ruta_archivo = os.path.join('Datasets Primer Parcial', '7-Statlog (Australian Credit Approval)', 'statlog+australian+credit+approval', 'australian.dat')

columnas = [f'A{i}' for i in range(1, 16)]
df = pd.read_csv(ruta_archivo, sep=r'\s+', header=None, names=columnas)

print('Dimensiones originales:', df.shape)
print('Primeras filas:')
display(df.head())
print('Tipos de dato:')
print(df.dtypes)
print('Nulos por columna:')
print(df.isnull().sum())

Dimensiones originales: (690, 15)
Primeras filas:


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15
0,1,22.08,11.46,2,4,4,1.585,0,0,0,1,2,100,1213,0
1,0,22.67,7.00,2,8,4,0.165,0,0,0,0,2,160,1,0
2,0,29.58,1.75,1,4,4,1.250,0,0,0,1,2,280,1,0
3,0,21.67,11.50,1,5,3,0.000,1,1,11,1,2,0,1,1
4,1,20.17,8.17,2,6,4,1.960,1,1,14,0,2,60,159,1


Tipos de dato:
A1       int64
A2     float64
A3     float64
A4       int64
A5       int64
A6       int64
A7     float64
A8       int64
A9       int64
A10      int64
A11      int64
A12      int64
A13      int64
A14      int64
A15      int64
dtype: object
Nulos por columna:
A1     0
A2     0
A3     0
A4     0
A5     0
A6     0
A7     0
A8     0
A9     0
A10    0
A11    0
A12    0
A13    0
A14    0
A15    0
dtype: int64


## Paso 2: Definicion de X e y y split 80/20
Se usa `A15` como variable objetivo y se divide en entrenamiento/prueba con el esquema clasico de cuadernillo usando indices aleatorios

In [62]:
import torch

target_col = 'A15'
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

torch.manual_seed(42)

n_total = X.shape[0]
n_test = int(0.2 * n_total)
n_train = n_total - n_test

indices = torch.randperm(n_total).tolist()
train_indices = indices[:n_train]
test_indices = indices[n_train:]

X_train_raw = X.iloc[train_indices].copy()
y_train_raw = y.iloc[train_indices].copy()
X_test_raw = X.iloc[test_indices].copy()
y_test_raw = y.iloc[test_indices].copy()

print(f'Train: {len(X_train_raw)}/{n_total} ({len(X_train_raw)/n_total*100:.1f}%)')
print(f'Test : {len(X_test_raw)}/{n_total} ({len(X_test_raw)/n_total*100:.1f}%)')
print('Distribucion en train (y):')
print(y_train_raw.value_counts().sort_index())
print('Distribucion en test (y):')
print(y_test_raw.value_counts().sort_index())
print('Shapes:', X_train_raw.shape, X_test_raw.shape, y_train_raw.shape, y_test_raw.shape)

Train: 552/690 (80.0%)
Test : 138/690 (20.0%)
Distribucion en train (y):
A15
0    309
1    243
Name: count, dtype: int64
Distribucion en test (y):
A15
0    74
1    64
Name: count, dtype: int64
Shapes: (552, 14) (138, 14) (552,) (138,)


## Paso 3: Manejo de nulos
En este dataset suele no haber nulos, pero se deja el paso completo: media en numericas y moda en categoricas.

In [63]:
# Segun la descripcion UCI: categoricas = A1, A4, A5, A6, A8, A9, A11, A12
categorical_cols = ['A1', 'A4', 'A5', 'A6', 'A8', 'A9', 'A11', 'A12']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print('Nulos antes (train):', X_train_raw.isnull().sum().sum())
print('Nulos antes (test) :', X_test_raw.isnull().sum().sum())

for col in numeric_cols:
    media = X_train_raw[col].mean()
    X_train_raw[col] = X_train_raw[col].fillna(media)
    X_test_raw[col] = X_test_raw[col].fillna(media)

for col in categorical_cols:
    moda = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(moda)
    X_test_raw[col] = X_test_raw[col].fillna(moda)

print('Nulos despues (train):', X_train_raw.isnull().sum().sum())
print('Nulos despues (test) :', X_test_raw.isnull().sum().sum())

Nulos antes (train): 0
Nulos antes (test) : 0
Nulos despues (train): 0
Nulos despues (test) : 0


## Paso 4: Funciones auxiliares
Se define `featureNormalize(X)` con el formato de los cuadernillos.

In [64]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])

    mu = np.mean(X, axis = 0)
    sigma = np.std(X, axis = 0)
    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## Paso 5: Codificacion y normalizacion
Como las categoricas ya vienen en codigos numericos (UCI), se usan como variables discretas y se normalizan todas las caracteristicas con `featureNormalize`.

In [65]:
target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(y_train_raw)
y_test = target_encoder.transform(y_test_raw)

X_train_mat = X_train_raw.to_numpy(dtype=float)
X_test_mat = X_test_raw.to_numpy(dtype=float)

X_train_norm, mu_norm, sigma_norm = featureNormalize(X_train_mat)
X_test_norm = (X_test_mat - mu_norm) / sigma_norm

X_train_final_df = pd.DataFrame(X_train_norm, columns=feature_cols, index=X_train_raw.index)
X_test_final_df = pd.DataFrame(X_test_norm, columns=feature_cols, index=X_test_raw.index)

print('X_train final:', X_train_final_df.shape)
print('X_test final :', X_test_final_df.shape)
display(X_train_final_df.head())

X_train final: (552, 14)
X_test final : (138, 14)


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14
522,-1.437591,-0.266303,-0.761256,0.541391,0.966527,-0.352703,-0.371633,0.971423,1.183216,-0.047933,1.103117,0.248069,-0.103807,-0.108530
607,0.695608,0.627262,-0.639077,0.541391,0.155291,-0.352703,-0.229747,0.971423,1.183216,1.580312,-0.906522,0.248069,0.015768,0.618838
686,0.695608,-0.922031,-0.875617,0.541391,0.155291,-0.352703,-0.619934,-1.029418,-0.845154,-0.454994,-0.906522,0.248069,-1.154806,-0.187502
313,-1.437591,-1.005776,-0.915692,0.541391,0.966527,1.628894,-0.573111,0.971423,1.183216,1.783842,-0.906522,0.248069,-0.651334,-0.177977
50,0.695608,0.054442,-0.272537,0.541391,-0.926357,-0.352703,-0.513519,-1.029418,-0.845154,-0.454994,1.103117,0.248069,0.305264,-0.195122


## Paso 6: Balanceo del train
Se aplica undersampling solamente sobre entrenamiento.

In [66]:
train_processed = X_train_final_df.copy()
train_processed[target_col] = y_train
test_processed = X_test_final_df.copy()
test_processed[target_col] = y_test

print('Distribucion antes del balanceo (train) - conteos:')
print(train_processed[target_col].value_counts().sort_index())
print('Distribucion antes del balanceo (train) - porcentajes:')
print((train_processed[target_col].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

min_count = train_processed[target_col].value_counts().min()
train_balanced = train_processed.groupby(target_col, group_keys=False).sample(n=min_count, random_state=42)

print('Distribucion despues del balanceo (train) - conteos:')
print(train_balanced[target_col].value_counts().sort_index())
print('Distribucion despues del balanceo (train) - porcentajes:')
print((train_balanced[target_col].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

print('Train balanceado:', train_balanced.shape)
print('Test sin balancear:', test_processed.shape)

Distribucion antes del balanceo (train) - conteos:
A15
0    309
1    243
Name: count, dtype: int64
Distribucion antes del balanceo (train) - porcentajes:
A15
0    55.98 %
1    44.02 %
Name: proportion, dtype: str
Distribucion despues del balanceo (train) - conteos:
A15
0    243
1    243
Name: count, dtype: int64
Distribucion despues del balanceo (train) - porcentajes:
A15
0    50.0 %
1    50.0 %
Name: proportion, dtype: str
Train balanceado: (486, 15)
Test sin balancear: (138, 15)


## Paso 7: Conversor a tensores (PyTorch)
En este paso se convierten los conjuntos preprocesados a tensores para usarlos directamente en modelos de PyTorch.

In [67]:
import torch

X_train_np = train_balanced.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_train_np = train_balanced[target_col].to_numpy().astype(np.int64)
X_test_np = test_processed.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_test_np = test_processed[target_col].to_numpy().astype(np.int64)

X_train = torch.from_numpy(X_train_np)
y_train = torch.from_numpy(y_train_np)
X_test = torch.from_numpy(X_test_np)
y_test = torch.from_numpy(y_test_np)

print('Preparación terminada:')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

Preparación terminada:
X_train shape: torch.Size([486, 14])
y_train shape: torch.Size([486])
X_test shape: torch.Size([138, 14])
y_test shape: torch.Size([138])
